# Device 3 Defect Localization with Unsupervised Clustering

This notebook is a teaching-focused, unsupervised defect-localization lab. The goal is not to produce perfect pixel masks. Instead, we use clustering to highlight **candidate defect regions** that look visually different from the rest of the image.

Important framing:
- Clustering is **unsupervised**, so it does not know the true defect pixels.
- A highlighted region is a **suspicious visual segment**, not a verified defect annotation.
- This works best when defects differ from the background in brightness, color, or texture.
- It works less well when lighting changes a lot or the normal surface is already complex.

The main walkthrough is built around **Device 3**. In the current repository snapshot, Device 3 has clean images but no matching defect-folder examples, so the notebook uses **Device 2 defect images as a fallback demonstration** while keeping the Device 3 clean reference workflow intact.


## 1. Imports and workshop helpers

We reuse the image-record loading helper so this notebook follows the same dataset conventions as the classification notebooks.


In [ ]:
# Standard library tools for paths and warning control.
from pathlib import Path
import sys
import warnings

# Data-science libraries for arrays, tables, plotting, and images.
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageFilter, ImageOps
from PIL.Image import Resampling
from IPython.display import Markdown, display

# Unsupervised models for region grouping and anomaly scoring.
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors

# Find the notebook folder so imports and dataset paths work from the repo root.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "0_Class_Notebooks":
    candidate = NOTEBOOK_DIR / "0_Class_Notebooks"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_image_utils import crop_dark_augmentation_edges, load_image_records

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## 2. Configuration

The notebook keeps the public configuration block small and explicit so students can safely experiment.

### Student change points

This notebook is especially good for experimentation because clustering results are visual. Change one setting at a time, rerun from this cell downward, and inspect the highlighted regions.

- `FOCUS_DEVICE`: choose the clean device used to build a reference.
- `IMAGE_SIZE`: smaller is faster; larger may preserve small defects.
- `COLOR_MODE`: compare `rgb` with `grayscale`.
- `CLEAN_REFERENCE_METHOD`: compare `median`, `mean`, `first`, and `middle`.
- `N_CLUSTERS`, `USE_POSITION_FEATURES`, and `POSITION_WEIGHT`: change what KMeans is allowed to group.
- `ANOMALY_RULE`: switch the rule used to decide what looks suspicious.

After each change, ask: did the model find a likely defect, or did it mostly find lighting, background, or edges?


In [ ]:
SOURCE_DATASET_ROOT = (NOTEBOOK_DIR.parent / "Device_Images_Nelson").resolve()
AUGMENTED_DATASET_ROOT = (NOTEBOOK_DIR.parent / "Augmented_Defect_Localization_Images").resolve()
CLEAN_REFERENCE_ROOT = AUGMENTED_DATASET_ROOT / "clean_references"
DATASET_ROOT = AUGMENTED_DATASET_ROOT if (AUGMENTED_DATASET_ROOT / "metadata.csv").exists() else SOURCE_DATASET_ROOT
METADATA_PATH = DATASET_ROOT / "metadata.csv"
################# Shared Hyperparameters ##################################
FOCUS_DEVICE = "device 3"  # Student change point: choose which clean device builds the reference image.
DEMO_DEFECT_DEVICE = "device 2"  # Student change point: fallback defect examples when the focus device has none.
IMAGE_SIZE = 100  # Student change point: smaller is faster; larger may preserve small defects.
COLOR_MODE = "rgb"  # Student change point: try "grayscale" if brightness matters more than color.
BLUR_RADIUS = 0.0  # Student change point: higher blur smooths noise but can hide tiny defects.
CLEAN_REFERENCE_METHOD = "median"  # Student change point: try "mean", "first", "middle", "index", or "path".
CLEAN_REFERENCE_IMAGE_INDEX = 0  # Used only when CLEAN_REFERENCE_METHOD = "index".
CLEAN_REFERENCE_IMAGE_PATH = None  # Used only when CLEAN_REFERENCE_METHOD = "path". Paste any image path here.
ALL_DEVICES_REFERENCE_METHOD = "median"  # Used by the all-devices extension; avoids reusing one custom path for every device.
PRIMARY_METHOD = "kmeans"  # Student change point: compare the primary method with later alternatives.
############### KMeans Hyperparameters ##################################
N_CLUSTERS = 3 # Student change point: try 2, 4, or 5 and inspect whether regions become more meaningful.

############ DBSCAN Hyperparameters ##################################
DBSCAN_EPS = 0.055 # Student change point: larger values merge more pixels; smaller values find tighter groups.
DBSCAN_MIN_SAMPLES = 4 # Student change point: lower values allow tiny clusters but may create false positives.
RUN_DBSCAN_COMPARISON = True  # DBSCAN can crash notebook kernels; opt in only on a strong machine.
MAX_DBSCAN_PIXELS = 20_000  # Hard safety cap for DBSCAN's pixel-neighborhood graph.
DBSCAN_WORKING_SIZE = 128  # Smaller than sqrt(MAX_DBSCAN_PIXELS), giving DBSCAN about 16k pixels.
DBSCAN_SVD_RANK = 16  # Set to None to disable low-rank SVD preprocessing for DBSCAN.

################### Nearest-Neighbor Hyperparameters ##################################
NN_ANOMALY_PERCENTILE = 92 # Student change point: higher means fewer pixels are flagged as suspicious.
USE_POSITION_FEATURES = True # Student change point: set False to group by image/color values without location.
POSITION_WEIGHT = 0.35 # Student change point: higher values make pixel location matter more.
ANOMALY_RULE = "deviation_from_clean_reference" # Student change point: try other listed rules and compare masks.
SUSPICIOUS_COLOR = (1.0, 0.05, 0.05)  # Red is the only color that means suspicious.


MASK_CMAP = ListedColormap(["#111111", "#ff1f1f"])
CLUSTER_CMAP = "tab20" # Good categorical colormap for up to 20 clusters; adjust if you use more clusters.
DISTANCE_CMAP = "magma" # Good sequential colormap for distance scores; adjust if you use a different scoring method that benefits from a different style of colormap.
MAX_EXAMPLES_PER_SECTION = 6 # Limit the number of examples shown in each section of the notebook to keep it concise and focused; set to None to show all examples.
RUN_ALL_DEVICES_EXTENSION = True # Whether to run the all-devices extension section at the end of the notebook, which applies the same methods but uses a reference image computed across all devices instead of per-device references; set to False to skip that section and save time.
RANDOM_STATE = 42

print(f"Source dataset root: {SOURCE_DATASET_ROOT}")
print(f"Augmented localization root: {AUGMENTED_DATASET_ROOT}")
print(f"Clean reference root: {CLEAN_REFERENCE_ROOT}")
print(f"Active dataset root: {DATASET_ROOT}")
print(f"Using augmented localization data: {DATASET_ROOT == AUGMENTED_DATASET_ROOT}")
print(f"Metadata file exists: {METADATA_PATH.exists()}")
print(f"Focus device: {FOCUS_DEVICE}")
print(f"Fallback defect device: {DEMO_DEFECT_DEVICE}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Color mode: {COLOR_MODE}")
print(f"Clean reference method: {CLEAN_REFERENCE_METHOD}")
print(f"Clean reference image path: {CLEAN_REFERENCE_IMAGE_PATH}")
print(f"All-devices reference method: {ALL_DEVICES_REFERENCE_METHOD}")
print(f"Primary method: {PRIMARY_METHOD}")
print(f"Clusters per image: {N_CLUSTERS}")
print(f"Run DBSCAN comparison: {RUN_DBSCAN_COMPARISON}")
print(f"DBSCAN eps: {DBSCAN_EPS}")
print(f"DBSCAN min_samples: {DBSCAN_MIN_SAMPLES}")
print(f"Max DBSCAN pixels: {MAX_DBSCAN_PIXELS:,}")
print(f"DBSCAN working size override: {DBSCAN_WORKING_SIZE}")
print(f"DBSCAN SVD rank: {DBSCAN_SVD_RANK}")
print(f"Nearest-neighbor anomaly percentile: {NN_ANOMALY_PERCENTILE}")
print(f"Use position features: {USE_POSITION_FEATURES}")
print(f"Anomaly rule: {ANOMALY_RULE}")
print(f"Examples shown per section: {MAX_EXAMPLES_PER_SECTION}")
print(f"Run all-devices extension: {RUN_ALL_DEVICES_EXTENSION}")

## 3. Load and filter the dataset

We load the image records, normalize device labels, and separate clean images from defect-folder images. The main walkthrough is still organized around Device 3 even if we need a fallback device for defect demonstrations.


In [ ]:
def normalize_device_name(value):
    return str(value).strip().lower()


def clean_reference_folder_to_device_type(folder_name):
    normalized = str(folder_name).strip().lower().replace("-", "_").replace(" ", "_")
    if normalized.startswith("device_"):
        suffix = normalized.split("device_", 1)[1]
        return "device1" if suffix == "1" else f"device {suffix}"
    if normalized.startswith("device"):
        suffix = normalized.split("device", 1)[1].strip("_ ")
        return "device1" if suffix == "1" else f"device {suffix}" if suffix else str(folder_name)
    return str(folder_name)


def load_clean_reference_records(clean_reference_root, dataset_root):
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    if not clean_reference_root.exists():
        return pd.DataFrame()

    rows = []
    for image_path in sorted(clean_reference_root.rglob("*")):
        if not image_path.is_file() or image_path.suffix.lower() not in valid_extensions:
            continue
        rel_path = image_path.relative_to(dataset_root)
        folder_parts = image_path.relative_to(clean_reference_root).parts
        device_folder = folder_parts[0] if folder_parts else image_path.parent.name
        device_type = clean_reference_folder_to_device_type(device_folder)
        rows.append(
            {
                "filepath": str(image_path),
                "relative_path": str(rel_path).replace("\\", "/"),
                "device_type": device_type,
                "damage_status": "Not-Damaged",
                "top_level_folder": "clean_references",
                "has_damage_label": True,
                "has_device_type_label": True,
                "device_type_label_source": "curated-clean-reference-folder",
                "is_defect_folder": False,
                "is_standard_device_image": False,
                "is_defect_challenge": False,
                "is_augmented": False,
                "source_relative_path": str(rel_path).replace("\\", "/"),
                "is_clean_reference": True,
            }
        )
    return pd.DataFrame(rows)


def clean_reference_mask(frame):
    if "is_clean_reference" in frame.columns:
        curated = frame["is_clean_reference"].fillna(False).astype(bool)
    else:
        curated = pd.Series(False, index=frame.index)

    if "damage_status" in frame.columns:
        explicit_not_damaged = frame["damage_status"].astype(str).str.strip().str.lower().eq("not-damaged")
    else:
        explicit_not_damaged = pd.Series(False, index=frame.index)

    return curated | explicit_not_damaged


def select_clean_reference_records(frame):
    if "is_clean_reference" in frame.columns:
        curated = frame[frame["is_clean_reference"].fillna(False).astype(bool)].reset_index(drop=True)
        if not curated.empty:
            return curated
    return frame[clean_reference_mask(frame)].reset_index(drop=True)


all_records = load_image_records(DATASET_ROOT, metadata_path=METADATA_PATH if METADATA_PATH.exists() else None)
all_records["is_clean_reference"] = False

clean_reference_records = load_clean_reference_records(CLEAN_REFERENCE_ROOT, DATASET_ROOT)
if not clean_reference_records.empty:
    all_records = pd.concat([all_records, clean_reference_records], ignore_index=True, sort=False)
    display(Markdown(
        f"**Loaded {len(clean_reference_records)} curated clean reference images** from `{CLEAN_REFERENCE_ROOT.name}`. "
        "These images are preferred when building per-device clean references."
    ))

records = all_records.copy()
records["device_key"] = records["device_type"].astype(str).map(normalize_device_name)

focus_key = normalize_device_name(FOCUS_DEVICE)
fallback_defect_key = normalize_device_name(DEMO_DEFECT_DEVICE)

focus_records = records[records["device_key"] == focus_key].reset_index(drop=True)
focus_clean_records = select_clean_reference_records(focus_records)
focus_defect_records = focus_records[focus_records["is_defect_folder"].fillna(False)].reset_index(drop=True)

fallback_defect_records = records[
    (records["device_key"] == fallback_defect_key) & records["is_defect_folder"].fillna(False)
].reset_index(drop=True)

if not focus_defect_records.empty:
    defect_demo_records = focus_defect_records.copy()
    defect_demo_device = FOCUS_DEVICE
    defect_demo_is_fallback = False
else:
    defect_demo_records = fallback_defect_records.copy()
    defect_demo_device = DEMO_DEFECT_DEVICE
    defect_demo_is_fallback = True

device_summary = (
    records.groupby("device_type", dropna=False)
    .agg(
        total_images=("filepath", "count"),
        clean_reference_images=("is_clean_reference", "sum"),
        not_damaged_images=("damage_status", lambda values: values.astype(str).str.strip().str.lower().eq("not-damaged").sum()),
        standard_images=("is_standard_device_image", "sum"),
        defect_folder_images=("is_defect_folder", "sum"),
    )
    .reset_index()
)

display(device_summary)
print(f"Focus device rows: {len(focus_records)}")
print(f"Focus device clean reference rows: {len(focus_clean_records)}")
print(f"Focus device defect rows: {len(focus_defect_records)}")
print(f"Defect demo rows: {len(defect_demo_records)} from {defect_demo_device}")

if focus_records.empty:
    raise ValueError(
        f"No records were found for {FOCUS_DEVICE}. Update FOCUS_DEVICE or point DATASET_ROOT to a dataset that contains it."
    )

if focus_clean_records.empty:
    raise ValueError(
        f"No clean reference images were found for {FOCUS_DEVICE}. Add curated images under `{CLEAN_REFERENCE_ROOT}` "
        "or provide metadata rows with damage_status = 'Not-Damaged'."
    )

if defect_demo_is_fallback:
    display(
        Markdown(
            f"**Note:** `{FOCUS_DEVICE}` currently has no defect-folder examples in `{DATASET_ROOT.name}`. "
            f"The notebook will still build a clean reference from `{FOCUS_DEVICE}`, but the defect visualization examples will use `{defect_demo_device}`."
        )
    )

## 4. Clustering idea: pixels plus position

A clustering model is not told where the defect is. It only sees pixel-level measurements and tries to group similar pixels together. After grouping, we decide which group looks most suspicious.

A plain pixel-only model can group visually similar pixels that are far apart from each other. To make the clusters more useful for localization, this notebook optionally includes each pixel's `(x, y)` position as part of the feature vector.

That means each pixel is described by:
- intensity or color
- horizontal position
- vertical position

This usually produces more spatially coherent regions, which makes the suspicious mask easier to interpret.

The models used later have different personalities:
- **KMeans** asks for a fixed number of groups, then assigns every pixel to the nearest group center. It is fast and predictable, but it will always force the image into exactly `N_CLUSTERS` regions.
- **DBSCAN** looks for dense neighborhoods of similar pixels. It can mark sparse pixels as noise, but it is much more memory-hungry because it has to reason about many pixel-to-pixel neighborhoods.
- **Nearest-neighbor anomaly scoring** compares the current image to the clean reference. It is not clustering, but it answers a useful question: which pixels look least like normal clean-device pixels?

Color guide:
- **Red always means suspicious** in the mask and overlay panels.
- Dark/black in the suspicious-mask panel means not selected as suspicious.
- Cluster-map colors are just category labels. A blue, green, purple, or yellow cluster is not automatically good or bad.
- In the nearest-neighbor distance map, brighter colors mean farther from the clean reference. The red overlay still marks the pixels selected as suspicious.


In [ ]:
def load_resized_image(image_path, image_size=IMAGE_SIZE, color_mode=COLOR_MODE, blur_radius=BLUR_RADIUS):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = crop_dark_augmentation_edges(image)
        image = ImageOps.fit(image, (image_size, image_size), method=Image.Resampling.BICUBIC)
        if blur_radius and blur_radius > 0:
            image = image.filter(ImageFilter.GaussianBlur(radius=blur_radius))
        if color_mode.lower() == "grayscale":
            image = image.convert("L")
        return np.asarray(image, dtype=np.float32) / 255.0


def build_clean_reference(
    clean_records,
    method=CLEAN_REFERENCE_METHOD,
    image_index=CLEAN_REFERENCE_IMAGE_INDEX,
    image_path=CLEAN_REFERENCE_IMAGE_PATH,
):
    method = str(method).strip().lower()
    if method == "path":
        if image_path is None or str(image_path).strip() == "":
            raise ValueError("Set CLEAN_REFERENCE_IMAGE_PATH when CLEAN_REFERENCE_METHOD = 'path'.")
        reference_path = Path(image_path).expanduser().resolve()
        if not reference_path.exists():
            raise FileNotFoundError(f"Clean reference image does not exist: {reference_path}")
        reference = load_resized_image(reference_path)
        return reference.astype(np.float32), f"custom image path: {reference_path.name}", 1

    clean_images = [load_resized_image(path) for path in clean_records["filepath"]]
    if not clean_images:
        raise ValueError("At least one clean image is required to build a clean reference.")

    stacked = np.stack(clean_images, axis=0)

    if method == "mean":
        reference = stacked.mean(axis=0)
        description = "mean of all clean images"
    elif method == "median":
        reference = np.median(stacked, axis=0).astype(np.float32)
        description = "median of all clean images"
    elif method == "first":
        reference = stacked[0]
        description = "first clean image"
    elif method == "middle":
        middle_index = len(clean_images) // 2
        reference = stacked[middle_index]
        description = f"middle clean image at index {middle_index}"
    elif method == "index":
        safe_index = int(np.clip(image_index, 0, len(clean_images) - 1))
        reference = stacked[safe_index]
        description = f"clean image at index {safe_index}"
    else:
        raise ValueError("CLEAN_REFERENCE_METHOD must be one of: mean, median, first, middle, index, path.")

    return reference.astype(np.float32), description, len(clean_images)


def intensity_view(image_array):
    if image_array.ndim == 2:
        return image_array
    return image_array.mean(axis=2)


def build_feature_matrix(image_array, use_position_features=USE_POSITION_FEATURES, position_weight=POSITION_WEIGHT):
    height, width = image_array.shape[:2]
    if image_array.ndim == 2:
        base_features = image_array.reshape(-1, 1)
    else:
        base_features = image_array.reshape(-1, image_array.shape[2])

    if not use_position_features:
        return base_features

    yy, xx = np.mgrid[0:height, 0:width]
    position_features = np.stack(
        [yy / max(height - 1, 1), xx / max(width - 1, 1)],
        axis=-1,
    ).reshape(-1, 2)
    return np.hstack([base_features, position_features * position_weight])


def normalize_cluster_labels(raw_labels):
    unique_labels = sorted(np.unique(raw_labels).tolist())
    label_map = {label: idx for idx, label in enumerate(unique_labels)}
    normalized = np.vectorize(label_map.get)(raw_labels)
    return normalized, label_map


def resize_array_image(image_array, target_size, resample=Resampling.BICUBIC):
    working = np.clip(image_array * 255.0, 0, 255).astype(np.uint8)
    mode = "L" if image_array.ndim == 2 else "RGB"
    pil_image = Image.fromarray(working, mode=mode)
    resized = pil_image.resize((target_size, target_size), resample=resample)
    return np.asarray(resized, dtype=np.float32) / 255.0


def resize_label_map(label_map, target_shape):
    pil_image = Image.fromarray(label_map.astype(np.int32), mode="I")
    resized = pil_image.resize((target_shape[1], target_shape[0]), resample=Resampling.NEAREST)
    return np.asarray(resized, dtype=np.int32)


def low_rank_svd_image(image_array, rank=DBSCAN_SVD_RANK):
    if rank is None or rank <= 0:
        return image_array

    def compress_channel(channel):
        safe_rank = max(1, min(rank, min(channel.shape)))
        u, s, vt = np.linalg.svd(channel, full_matrices=False)
        return (u[:, :safe_rank] * s[:safe_rank]) @ vt[:safe_rank, :]

    if image_array.ndim == 2:
        compressed = compress_channel(image_array)
    else:
        compressed = np.stack([compress_channel(image_array[..., idx]) for idx in range(image_array.shape[2])], axis=-1)
    return np.clip(compressed, 0.0, 1.0)


def prepare_dbscan_image(image_array, max_pixels=MAX_DBSCAN_PIXELS, working_size=DBSCAN_WORKING_SIZE, svd_rank=DBSCAN_SVD_RANK):
    safe_side = int(np.sqrt(max_pixels))
    current_side = image_array.shape[0]
    target_side = min(current_side, working_size or safe_side, safe_side)
    prepared = image_array
    resized = False

    if current_side > target_side:
        prepared = resize_array_image(prepared, target_side)
        resized = True

    prepared = low_rank_svd_image(prepared, rank=svd_rank)
    pixel_count = int(np.prod(prepared.shape[:2]))
    if pixel_count > max_pixels:
        raise ValueError(
            f"DBSCAN still has {pixel_count:,} pixels after preprocessing. "
            f"Lower IMAGE_SIZE or DBSCAN_WORKING_SIZE below {safe_side}."
        )

    return prepared, {
        "original_shape": image_array.shape[:2],
        "working_shape": prepared.shape[:2],
        "resized_for_dbscan": resized,
        "svd_rank_used": svd_rank,
        "safe_side": safe_side,
    }


def cluster_image_kmeans(image_array, n_clusters=N_CLUSTERS, use_position_features=USE_POSITION_FEATURES, position_weight=POSITION_WEIGHT):
    features = build_feature_matrix(image_array, use_position_features=use_position_features, position_weight=position_weight)
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(features).reshape(image_array.shape[:2])
    return labels, model


def cluster_image_dbscan(image_array, eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, use_position_features=USE_POSITION_FEATURES, position_weight=POSITION_WEIGHT):
    working_image, prep_info = prepare_dbscan_image(image_array)
    features = build_feature_matrix(working_image, use_position_features=use_position_features, position_weight=position_weight)
    model = DBSCAN(eps=eps, min_samples=min_samples)
    raw_labels = model.fit_predict(features).reshape(working_image.shape[:2])
    labels, label_map = normalize_cluster_labels(raw_labels)
    display_labels = resize_label_map(labels, image_array.shape[:2])
    return labels, model, raw_labels, label_map, working_image, prep_info, display_labels


def nearest_neighbor_anomaly_map(image_array, clean_reference, use_position_features=USE_POSITION_FEATURES, position_weight=POSITION_WEIGHT):
    image_features = build_feature_matrix(image_array, use_position_features=use_position_features, position_weight=position_weight)
    reference_features = build_feature_matrix(clean_reference, use_position_features=use_position_features, position_weight=position_weight)
    neighbor_model = NearestNeighbors(n_neighbors=1)
    neighbor_model.fit(reference_features)
    distances, _ = neighbor_model.kneighbors(image_features)
    distance_map = distances.reshape(image_array.shape[:2])
    return distance_map, neighbor_model


def cluster_statistics(image_array, labels, clean_reference=None):
    image_intensity = intensity_view(image_array)
    reference_intensity = intensity_view(clean_reference) if clean_reference is not None else None
    counts = np.bincount(labels.ravel(), minlength=int(labels.max()) + 1)
    dominant_cluster = int(np.argmax(counts))
    dominant_mean = float(image_intensity[labels == dominant_cluster].mean())

    rows = []
    for cluster_id in range(len(counts)):
        mask = labels == cluster_id
        area_fraction = float(mask.mean())
        cluster_mean = float(image_intensity[mask].mean())
        contrast_to_dominant = abs(cluster_mean - dominant_mean)
        if reference_intensity is not None:
            deviation_from_reference = float(np.abs(image_intensity[mask] - reference_intensity[mask]).mean())
        else:
            deviation_from_reference = np.nan
        rows.append(
            {
                "cluster_id": cluster_id,
                "pixel_count": int(mask.sum()),
                "area_fraction": area_fraction,
                "cluster_mean_intensity": cluster_mean,
                "contrast_to_dominant": contrast_to_dominant,
                "deviation_from_clean_reference": deviation_from_reference,
                "is_dominant_cluster": cluster_id == dominant_cluster,
            }
        )

    stats = pd.DataFrame(rows)
    return stats, dominant_cluster


def choose_suspicious_cluster(stats, rule=ANOMALY_RULE):
    candidate_stats = stats[~stats["is_dominant_cluster"]].copy()
    if candidate_stats.empty:
        return int(stats.iloc[0]["cluster_id"]), stats.assign(score=0.0)

    if rule == "deviation_from_clean_reference" and candidate_stats["deviation_from_clean_reference"].notna().any():
        # Reward deviation from the clean baseline, but avoid always picking the tiniest cluster.
        candidate_stats["score"] = candidate_stats["deviation_from_clean_reference"] / np.sqrt(candidate_stats["area_fraction"].clip(lower=1e-4))
    elif rule == "smallest_high_contrast_cluster":
        contrast_threshold = candidate_stats["contrast_to_dominant"].median()
        filtered = candidate_stats[candidate_stats["contrast_to_dominant"] >= contrast_threshold].copy()
        if filtered.empty:
            filtered = candidate_stats.copy()
        filtered["score"] = filtered["contrast_to_dominant"] / filtered["area_fraction"].clip(lower=1e-4)
        candidate_stats = filtered
    else:
        candidate_stats["score"] = candidate_stats["contrast_to_dominant"] / np.sqrt(candidate_stats["area_fraction"].clip(lower=1e-4))

    suspicious_cluster = int(candidate_stats.sort_values("score", ascending=False).iloc[0]["cluster_id"])
    merged = stats.merge(candidate_stats[["cluster_id", "score"]], on="cluster_id", how="left")
    merged["score"] = merged["score"].fillna(0.0)
    return suspicious_cluster, merged.sort_values("score", ascending=False)


def cluster_map_rgb(labels):
    cmap = plt.cm.get_cmap(CLUSTER_CMAP, int(labels.max()) + 1)
    return cmap(labels / max(labels.max(), 1))[..., :3]


def overlay_mask(image_array, mask, color=SUSPICIOUS_COLOR, alpha=0.55):
    if image_array.ndim == 2:
        base = np.dstack([image_array] * 3)
    else:
        base = image_array.copy()
    overlay = base.copy()
    mask_bool = mask.astype(bool)
    overlay[mask_bool] = (1 - alpha) * overlay[mask_bool] + alpha * np.array(color)
    return np.clip(overlay, 0.0, 1.0)


def analyze_record(record, clean_reference=None, rule=ANOMALY_RULE, method=PRIMARY_METHOD):
    image_array = load_resized_image(record.filepath)
    if method == "kmeans":
        labels, model = cluster_image_kmeans(image_array)
        stats, dominant_cluster = cluster_statistics(image_array, labels, clean_reference=clean_reference)
        suspicious_cluster, ranked_stats = choose_suspicious_cluster(stats, rule=rule)
        suspicious_mask = labels == suspicious_cluster
        return {
            "record": record,
            "image_array": image_array,
            "labels": labels,
            "cluster_map": cluster_map_rgb(labels),
            "panel_image": cluster_map_rgb(labels),
            "panel_title": "Cluster map (KMeans)",
            "model": model,
            "dominant_cluster": dominant_cluster,
            "suspicious_cluster": suspicious_cluster,
            "suspicious_mask": suspicious_mask,
            "overlay": overlay_mask(image_array, suspicious_mask),
            "stats": ranked_stats,
            "suspicious_fraction": float(suspicious_mask.mean()),
            "rule": rule,
            "method": method,
        }

    if method == "dbscan":
        labels, model, raw_labels, label_map, working_image, prep_info, display_labels = cluster_image_dbscan(image_array)
        working_reference = None
        if clean_reference is not None:
            working_reference, _ = prepare_dbscan_image(clean_reference, svd_rank=prep_info["svd_rank_used"])
        stats, dominant_cluster = cluster_statistics(working_image, labels, clean_reference=working_reference)
        suspicious_cluster, ranked_stats = choose_suspicious_cluster(stats, rule=rule)
        suspicious_mask = display_labels == suspicious_cluster
        noise_fraction = float((raw_labels == -1).mean())
        ranked_stats["noise_fraction"] = noise_fraction
        ranked_stats["dbscan_working_side"] = int(prep_info["working_shape"][0])
        ranked_stats["dbscan_resized"] = bool(prep_info["resized_for_dbscan"])
        ranked_stats["dbscan_svd_rank"] = prep_info["svd_rank_used"] if prep_info["svd_rank_used"] is not None else np.nan
        panel_title = "Cluster map (DBSCAN)"
        if prep_info["resized_for_dbscan"] or prep_info["svd_rank_used"] is not None:
            panel_title += f" | working={prep_info['working_shape'][0]}"
            if prep_info["svd_rank_used"] is not None:
                panel_title += f" | svd={prep_info['svd_rank_used']}"
        return {
            "record": record,
            "image_array": image_array,
            "labels": display_labels,
            "cluster_map": cluster_map_rgb(display_labels),
            "panel_image": cluster_map_rgb(display_labels),
            "panel_title": panel_title,
            "model": model,
            "dominant_cluster": dominant_cluster,
            "suspicious_cluster": suspicious_cluster,
            "suspicious_mask": suspicious_mask,
            "overlay": overlay_mask(image_array, suspicious_mask),
            "stats": ranked_stats,
            "suspicious_fraction": float(suspicious_mask.mean()),
            "rule": rule,
            "method": method,
            "dbscan_prep": prep_info,
        }

    if method == "nearest_neighbor":
        if clean_reference is None:
            raise ValueError("Nearest-neighbor anomaly scoring needs a clean reference image.")
        distance_map, model = nearest_neighbor_anomaly_map(image_array, clean_reference)
        threshold = float(np.quantile(distance_map, NN_ANOMALY_PERCENTILE / 100.0))
        suspicious_mask = distance_map >= threshold
        stats = pd.DataFrame(
            {
                "metric": ["mean_distance", "max_distance", "threshold", "suspicious_fraction"],
                "value": [
                    float(distance_map.mean()),
                    float(distance_map.max()),
                    threshold,
                    float(suspicious_mask.mean()),
                ],
            }
        )
        return {
            "record": record,
            "image_array": image_array,
            "labels": None,
            "cluster_map": None,
            "panel_image": distance_map,
            "panel_title": "Distance map (Nearest Neighbor)",
            "model": model,
            "dominant_cluster": None,
            "suspicious_cluster": None,
            "suspicious_mask": suspicious_mask,
            "overlay": overlay_mask(image_array, suspicious_mask),
            "stats": stats,
            "suspicious_fraction": float(suspicious_mask.mean()),
            "rule": "nearest_neighbor_distance",
            "method": method,
        }

    raise ValueError(f"Unsupported method: {method}")


def show_analysis(result, title_prefix=""):
    record = result["record"]
    image_array = result["image_array"]
    suspicious_mask = result["suspicious_mask"]
    stats = result["stats"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    if image_array.ndim == 2:
        axes[0].imshow(image_array, cmap="gray")
    else:
        axes[0].imshow(image_array)
    axes[0].set_title("Original")

    if result["method"] == "nearest_neighbor":
        axes[1].imshow(result["panel_image"], cmap=DISTANCE_CMAP)
    else:
        axes[1].imshow(result["panel_image"])
    axes[1].set_title(result["panel_title"])

    axes[2].imshow(suspicious_mask.astype(int), cmap=MASK_CMAP, vmin=0, vmax=1)
    axes[2].set_title("Suspicious mask\nred = suspicious")

    axes[3].imshow(result["overlay"])
    axes[3].set_title("Overlay\nred = suspicious")

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        f"{title_prefix}{record.device_type} | {record.relative_path} | method={result['method']} | rule={result['rule']} | suspicious area={result['suspicious_fraction']:.3f}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

    if result["method"] == "nearest_neighbor":
        display(stats.round(4))
    else:
        columns = [column for column in ["cluster_id", "area_fraction", "contrast_to_dominant", "deviation_from_clean_reference", "score", "noise_fraction", "dbscan_working_side", "dbscan_resized", "dbscan_svd_rank"] if column in stats.columns]
        display(stats[columns].round(4))


## 5. Build a clean Device 3 reference

A clean reference is the notebook's picture of what this device usually looks like when no defect is present. It is built from curated images in `Augmented_Defect_Localization_Images/clean_references` when available, after the same resizing, edge cleanup, color conversion, and optional blur used everywhere else.

Why it helps:
- Clustering alone can find regions, but it does not know whether a region is normal for this device.
- Comparing each region to a clean reference gives the notebook a normal baseline.
- The suspicious score can then prefer clusters that differ from normal clean-device appearance, not just clusters that are small or high contrast.

You can control the reference with `CLEAN_REFERENCE_METHOD` in the configuration cell:
- `median`: default; a robust composite that ignores one-off clean-image oddities better than a mean.
- `mean`: a smooth average of all clean images; useful when clean images are well aligned and consistent.
- `first` or `middle`: use one real clean image instead of a composite.
- `index`: use a specific clean image chosen by `CLEAN_REFERENCE_IMAGE_INDEX`.
- `path`: use any image on your computer chosen by `CLEAN_REFERENCE_IMAGE_PATH`.

In [ ]:
clean_reference, clean_reference_description, clean_reference_image_count = build_clean_reference(focus_clean_records)

plt.figure(figsize=(5, 5))
if clean_reference.ndim == 2:
    plt.imshow(clean_reference, cmap="gray")
else:
    plt.imshow(clean_reference)
plt.title(f"Clean reference for {FOCUS_DEVICE}\n{clean_reference_description}")
plt.axis("off")
plt.show()

display(
    pd.DataFrame(
        {
            "focus_device": [FOCUS_DEVICE],
            "clean_reference_method": [CLEAN_REFERENCE_METHOD],
            "clean_reference_description": [clean_reference_description],
            "clean_reference_images": [clean_reference_image_count],
            "defect_images_for_focus_device": [len(focus_defect_records)],
            "defect_demo_device": [defect_demo_device],
        }
    )
)


## 6. Clean Device 3 examples

On a clean image, we usually hope the suspicious mask stays small or low-signal. It will not always disappear, because clustering still has to partition the image into regions.


In [ ]:
for record in focus_clean_records.head(MAX_EXAMPLES_PER_SECTION).itertuples(index=False):
    result = analyze_record(record, clean_reference=clean_reference, rule=ANOMALY_RULE, method=PRIMARY_METHOD)
    show_analysis(result, title_prefix="Clean example | ")


## 7. Defect-localization examples

This is the main visual deliverable of the notebook: the original image, the cluster map, the suspicious mask, and the overlay.

In every result below, **red is the only suspicious color**. The cluster-map colors are arbitrary region labels, so they should not be read as risk levels.

If the focus device has no defect-folder examples, the notebook uses a fallback defect device so the localization workflow can still be demonstrated.


In [ ]:
if defect_demo_records.empty:
    display(Markdown("**No defect examples available:** add defect images to the dataset to run this section."))
else:
    display(
        Markdown(
            f"**Defect demo device:** `{defect_demo_device}` | **Fallback in use:** `{defect_demo_is_fallback}` | **Method:** `{PRIMARY_METHOD}` | **Rule:** `{ANOMALY_RULE}`"
        )
    )
    for record in defect_demo_records.head(MAX_EXAMPLES_PER_SECTION).itertuples(index=False):
        result = analyze_record(record, clean_reference=clean_reference, rule=ANOMALY_RULE, method=PRIMARY_METHOD)
        show_analysis(result, title_prefix="Defect example | ")


## 8. Compare different unsupervised models

We compare three unsupervised approaches side by side. They are trying to answer related but slightly different questions.

- `KMeans`: What groups of pixels look similar if we force the image into `N_CLUSTERS` regions? This is the main teaching model because it is fast, stable, and easy to visualize.
- `DBSCAN`: Which pixels belong to dense neighborhoods, and which pixels look isolated enough to be noise? This can be useful for irregular defects, but it can be slow or memory-heavy on full-resolution pixel grids.
- `Nearest neighbor anomaly map`: How far is each pixel from the most similar clean-reference pixel? This is not a clustering model, but it directly visualizes deviation from normal.

This is also the right place to clarify a common confusion: **K-nearest neighbors (KNN)** is usually a supervised classifier, not a clustering algorithm. A nearest-neighbor anomaly score, however, works well as an unsupervised comparison against clustering.

Low-rank **SVD** is useful here mainly as a denoising step before DBSCAN. It smooths the image into its strongest patterns before neighborhood clustering. It does not reduce the number of pixels by itself, so we still resize to a safe working image when needed.


In [ ]:
if defect_demo_records.empty:
    display(Markdown("**Comparison skipped:** no defect examples are available."))
else:
    comparison_record = next(defect_demo_records.head(1).itertuples(index=False))
    comparison_settings = [
        ("kmeans", "deviation_from_clean_reference"),
    ]

    if RUN_DBSCAN_COMPARISON:
        comparison_settings.append(("dbscan", "deviation_from_clean_reference"))
    else:
        display(
            Markdown(
                "**DBSCAN skipped by default:** it can allocate a very large pixel-neighborhood graph and crash the notebook kernel. "
                "Set `RUN_DBSCAN_COMPARISON = True` in the configuration cell to opt in."
            )
        )

    comparison_settings.append(("nearest_neighbor", "nearest_neighbor_distance"))

    for method_name, rule_name in comparison_settings:
        try:
            result = analyze_record(comparison_record, clean_reference=clean_reference, rule=rule_name, method=method_name)
        except MemoryError:
            display(Markdown(f"**{method_name} skipped:** it ran out of memory. Try a smaller `IMAGE_SIZE` or `DBSCAN_WORKING_SIZE`."))
            continue
        except ValueError as error:
            display(Markdown(f"**{method_name} skipped:** {error}"))
            continue
        show_analysis(result, title_prefix="Model comparison | ")


## 9. Failure or hard-case example

A useful teaching notebook should show at least one case where the method is less convincing. Here we look for the clean image with the largest suspicious region fraction, which often behaves like a false positive or over-sensitive segmentation.


In [ ]:
clean_screening_results = [
    analyze_record(record, clean_reference=clean_reference, rule=ANOMALY_RULE, method=PRIMARY_METHOD)
    for record in focus_clean_records.head(min(len(focus_clean_records), 8)).itertuples(index=False)
]

if clean_screening_results:
    hardest_clean_result = max(clean_screening_results, key=lambda item: item["suspicious_fraction"])
    display(
        Markdown(
            "**Why this is a hard case:** the suspicious region covers a relatively large portion of a clean image, "
            "which means clustering is reacting to lighting or texture changes rather than a true known defect."
        )
    )
    show_analysis(hardest_clean_result, title_prefix="Hard case | ")


## 10. Optional all-devices extension

The first version of the notebook is intentionally centered on one device. This extension shows how the same logic could be wrapped across all devices without rewriting the pipeline.

Important: if the main walkthrough uses `CLEAN_REFERENCE_METHOD = "path"`, that custom image is only for the focus-device walkthrough. The all-devices extension builds a separate reference for each device using `ALL_DEVICES_REFERENCE_METHOD`, so Device 1 and Device 2 are not compared to a Device 3 reference by accident.

This section prefers images from `clean_references/<device folder>` as each device's clean reference set. If a device has no curated clean-reference folder, it falls back only to explicit `damage_status == "Not-Damaged"` metadata. Folder location alone is not enough, because a device folder can contain images with visible defects.

In [ ]:
if not RUN_ALL_DEVICES_EXTENSION:
    display(Markdown("**Extension skipped:** set `RUN_ALL_DEVICES_EXTENSION = True` to run the cross-device summary."))
else:
    extension_rows = []
    for device_name in sorted(records["device_type"].astype(str).unique().tolist()):
        device_key = normalize_device_name(device_name)
        device_records = records[records["device_key"] == device_key].reset_index(drop=True)
        clean_subset = select_clean_reference_records(device_records)
        defect_subset = device_records[device_records["is_defect_folder"].fillna(False)].reset_index(drop=True)

        if clean_subset.empty:
            extension_rows.append(
                {
                    "device_type": device_name,
                    "clean_images": 0,
                    "defect_images": int(len(defect_subset)),
                    "avg_clean_suspicious_fraction": np.nan,
                    "avg_defect_suspicious_fraction": np.nan,
                    "status": "No clean reference images available",
                }
            )
            continue

        device_reference, _, _ = build_clean_reference(clean_subset, method=ALL_DEVICES_REFERENCE_METHOD, image_path=None)
        clean_results = [
            analyze_record(record, clean_reference=device_reference, rule=ANOMALY_RULE, method=PRIMARY_METHOD)
            for record in clean_subset.head(min(len(clean_subset), 3)).itertuples(index=False)
        ]
        defect_results = [
            analyze_record(record, clean_reference=device_reference, rule=ANOMALY_RULE, method=PRIMARY_METHOD)
            for record in defect_subset.head(min(len(defect_subset), 3)).itertuples(index=False)
        ]

        extension_rows.append(
            {
                "device_type": device_name,
                "clean_images": int(len(clean_subset)),
                "defect_images": int(len(defect_subset)),
                "avg_clean_suspicious_fraction": float(np.mean([item["suspicious_fraction"] for item in clean_results])) if clean_results else np.nan,
                "avg_defect_suspicious_fraction": float(np.mean([item["suspicious_fraction"] for item in defect_results])) if defect_results else np.nan,
                "status": "OK" if defect_results else "No defect examples for this device",
            }
        )

    extension_summary = pd.DataFrame(extension_rows)
    display(extension_summary.round(4))

## 11. What students should notice

- Clustering gives **candidate regions**, not verified defect masks.
- Adding pixel position helps the segments stay spatially coherent.
- Comparing clusters to a clean reference is usually more persuasive than clustering a defect image in isolation.
- A clean image can still produce a suspicious mask, which is an important reminder that unsupervised localization is approximate.
- If you later want a true supervised defect-localization notebook, you will need pixel masks or bounding boxes.
